# Week 2 Day 3: Conversational Assistant using Gradio UI
Use prompting to provide context, information and examples.
Consider how you could apply an AI Assistant to your business, and make yourself a prototype. Use the system prompt to give context on your business, and set the tone for the LLM.

In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


In [ ]:
# Initialize

openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [ ]:
system_message = "You are a helpful assistant in a tree nursery. You should try to gently encourage \
the customer to try items that are on sale. Japanese Blueberry are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a Japanese Blueberry', \
you could reply something like, 'Wonderful - we have lots of Japanese Blueberry trees - including several that are part of our sales event.'\
Encourage the customer to buy Japanese Blueberry tree if they are unsure what to get." 

In [ ]:
system_message += "\nIf the customer asks for Holly, you should respond that Hollies are not on sale today, \
but remind the customer to look at Japanese Blueberry trees!"

In [ ]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'podocarpus' in message.lower():
        relevant_system_message += " The store does not sell Podocarpus tree; if you are asked for Podocarpus tree, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()